Mini-Project 1: Brazilian E-Commerce

1. Load and Explore the Data : We will use two libraries in Python to load and explore our dataset: pandas and sqlalchemy.

2. Create SQLite Database and Export DataFrames.

    Create a SQLite database using SQLAlchemy.
    Export each dataframe as a table to the SQLite database.


In [ ]:
import numpy as np # linear algebra
import pandas as pd
import sqlite3


# Read all CSV files from the same consistent path
data_path = 'kaggle/input/brazilian-ecommerce/'

df_olist_customers = pd.read_csv(f'{data_path}olist_customers_dataset.csv')
df_olist_sellers = pd.read_csv(f'{data_path}olist_sellers_dataset.csv')
df_olist_order_reviews= pd.read_csv(f'{data_path}olist_order_reviews_dataset.csv')
df_olist_order_items= pd.read_csv(f'{data_path}olist_order_items_dataset.csv')
df_olist_products= pd.read_csv(f'{data_path}olist_products_dataset.csv')
df_olist_geolocation= pd.read_csv(f'{data_path}olist_geolocation_dataset.csv')
df_product_category_name_translation= pd.read_csv(f'{data_path}product_category_name_translation.csv')
df_olist_orders = pd.read_csv(f'{data_path}olist_orders_dataset.csv')
df_olist_order_payments= pd.read_csv(f'{data_path}olist_order_payments_dataset.csv')

# df_olist_customers = pd.read_csv('kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv')
# df_olist_sellers = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_sellers_dataset.csv')
# df_olist_order_reviews= pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv')
# df_olist_order_items= pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv')
# df_olist_products= pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_products_dataset.csv')
# df_olist_geolocation= pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_geolocation_dataset.csv')
# df_product_category_name_translation= pd.read_csv('/kaggle/input/brazilian-ecommerce/product_category_name_translation.csv')
# df_olist_orders = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv')
# df_olist_order_payments= pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_order_payments_dataset.csv')

df_olist_customers.head()

from sqlalchemy import create_engine
engine = create_engine('sqlite:///brazil-ecom.db', echo=False)

# export the dataframe as a table 'playstore' to the sqlite engine
df_olist_customers.to_sql("olist_customers", con =engine)
df_olist_sellers.to_sql("olist_sellers", con =engine)
df_olist_order_reviews.to_sql("olist_order_reviews", con =engine)
df_olist_order_items.to_sql("olist_order_items", con =engine)
df_olist_products.to_sql("olist_products_dataset", con =engine)
df_olist_geolocation.to_sql("olist_geolocation", con =engine)
df_product_category_name_translation.to_sql("product_category_name_translation", con =engine)
df_olist_orders.to_sql("olist_orders", con =engine)
df_olist_order_payments.to_sql("olist_order_payments", con =engine)
df_olist_order_payments.head()

In [2]:
sql='''

Select * from olist_customers
limit 5


''';


df_sql = pd.read_sql_query(sql,con=engine)
df_sql.head()

,index,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [3]:
# 3. Query 1: Count and Percentage of Orders Purchased in Jan 2018 with 5 Review Score

#     Write and execute a SQL query to count the number of orders purchased in January 2018 that have a review score of 5 and calculate the percentage of such orders.


sql='''
SELECT (
    (SELECT COUNT(DISTINCT oo_inner.order_id)
     FROM olist_orders oo_inner
     JOIN olist_order_reviews oor ON oo_inner.order_id = oor.order_id
     WHERE strftime('%Y-%m', oo_inner.order_purchase_timestamp) = '2018-01' 
     AND oor.review_score = 5) 
    * 1.0 /  -- This forces the division to be decimal
    COUNT(DISTINCT oo.order_id)
) * 100 AS jan18_5_review_count_percentage
FROM olist_orders oo''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,jan18_5_review_count_percentage
0,4.099919


In [4]:
     

# 4. Query 2: Customer Purchase Trend Year-on-Year

#     Write and execute a SQL query to analyze the customer purchase trend year-on-year.
# Must use the unique customer id, not the order customer id
sql='''
SELECT 
    strftime('%Y', order_purchase_timestamp) AS purchase_year,
    COUNT(order_id) AS total_orders,
    COUNT(DISTINCT oc.customer_unique_id) AS unique_customers
FROM olist_orders oo
JOIN olist_customers oc ON oo.customer_id = oc.customer_id
GROUP BY purchase_year
ORDER BY purchase_year;
''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,purchase_year,total_orders,unique_customers
0,2016,329,326
1,2017,45101,43713
2,2018,54011,52749


In [5]:
# 5. Query 3: Average Order Values of Customers

#     Write and execute a SQL query to calculate the average order values of customers.
# To find an order value, actually need a subquery to create the sum of payment values for each order.

#One way to find total order value (needs subquery):

# sql='''
# SELECT oop.order_id, SUM(oop.payment_value) AS order_value
# FROM olist_orders oo
# JOIN olist_order_payments oop ON oo.order_id = oop.order_id
# GROUP BY oop.order_id
# ORDER BY order_value DESC

# ;
# ''';

# Another way is SUM(ooi.price + ooi.freight_value), GROUP BY order_id. Also requires subquery:

# sql='''
# SELECT ooi.order_id, SUM(ooi.price + ooi.freight_value) AS order_value
# FROM olist_orders oo
# JOIN olist_order_items ooi ON oo.order_id = ooi.order_id
# GROUP BY ooi.order_id
# ORDER BY order_value DESC

#  ;
#  ''';

#Bringing it together in a subquery (remember to group by customer unique IDs):

sql='''
WITH order_values AS (SELECT ooi.order_id, SUM(ooi.price + ooi.freight_value) AS order_value
FROM olist_orders oo
JOIN olist_order_items ooi ON oo.order_id = ooi.order_id
GROUP BY ooi.order_id
ORDER BY order_value DESC)
SELECT oc.customer_unique_id, AVG(ov.order_value) AS avg_order_value
FROM olist_orders oo
JOIN order_values ov ON oo.order_id = ov.order_id
JOIN olist_customers oc ON oo.customer_id = oc.customer_id
GROUP BY oc.customer_unique_id

 ;
 ''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql.head()


,customer_unique_id,avg_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89


In [6]:
# 6. Query 4: Top 5 Cities with Highest Revenue from 2016 to 2018

#     Write and execute a SQL query to find the top 5 cities with the highest revenue from 2016 to 2018.

# [[Strategy: How to calculate revenue: SUM(oop.payment_value) with JOIN olist_customers oc.customer_id	

# How to do top 5 cities per year: use window functions? group by? GROUP BY oc.customer_city, year_calculation
# Use GROUP BY to create revenue, and RANK window function]]

sql='''
WITH yr_city_ranks AS (
SELECT oc.customer_city, strftime('%Y', oo.order_purchase_timestamp) AS year, SUM(oop.payment_value) AS revenue,
RANK() OVER(PARTITION BY strftime('%Y', oo.order_purchase_timestamp) ORDER BY SUM(oop.payment_value) DESC) AS rank
FROM olist_orders oo
JOIN olist_order_payments oop ON oo.order_id = oop.order_id
JOIN olist_customers oc ON oo.customer_id = oc.customer_id
GROUP BY oc.customer_city, strftime('%Y', oo.order_purchase_timestamp))
SELECT * FROM yr_city_ranks
WHERE rank < 6
;
''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,customer_city,year,revenue,rank
0,rio de janeiro,2016,8848.50,1
1,sao paulo,2016,4724.49,2
2,belo horizonte,2016,1741.47,3
3,quissama,2016,1400.74,4
4,porto alegre,2016,1276.09,5
5,sao paulo,2017,909863.89,1
6,rio de janeiro,2017,574541.77,2
7,belo horizonte,2017,179560.44,3
8,brasilia,2017,156165.53,4
9,porto alegre,2017,109667.77,5


In [8]:

# 7. Query 5: State Wise Revenue Table Between 2016 to 2018

#     Write and execute a SQL query to create a state-wise revenue table between 2016 to 2018.

# similar strategy to above, using customer.customer_state, no rank.
sql='''
SELECT oc.customer_state, strftime('%Y', oo.order_purchase_timestamp) AS year, SUM(oop.payment_value) AS revenue
FROM olist_orders oo
JOIN olist_order_payments oop ON oo.order_id = oop.order_id
JOIN olist_customers oc ON oo.customer_id = oc.customer_id
GROUP BY oc.customer_state, strftime('%Y', oo.order_purchase_timestamp)

;
''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,customer_state,year,revenue
0,AC,2017,12997.52
1,AC,2018,6683.10
2,AL,2016,129.90
3,AL,2017,52294.67
4,AL,2018,44537.49
...,...,...,...
70,SP,2016,16885.54
71,SP,2017,2561862.91
72,SP,2018,3419478.51
73,TO,2017,28827.94


In [9]:
# 8. Query 6: Top Successful Sellers in Terms of Goods Sold, Revenue, and Customer Count

#     Write and execute a SQL query to identify the top successful sellers in terms of the number of goods sold, total revenue, customer count, and sellers with the highest 5-star ratings.

# The instructions are vague here. It's unclear whether the query should show a specific number of top sellers and whether it should show top sellers for each category or simply aggregate. I've decided to show all these categories and aggregate, ordering by the various categories. The query ranks all sellers and then one can find the top sellers within the result table.

# Strategy: Table w many joins, group by seller ID. 
# number of goods sold = total number of items sold. So simply COUNT ooi.product_id
# Revenue = SUM(oop.payment_value) AS revenue;
# customer count = COUNT(DISTNICT oc.customer_unique_id)
# highest 5-star ratings: SUM(CASE WHEN oor.review_score=5 THEN 1 END) [JOIN olist_order_reviews oor ON oo.order_id = oor.order_id]

sql='''
SELECT ooi.seller_id,
COUNT(ooi.product_id) AS goods_sold,
SUM(oop.payment_value) AS revenue,
COUNT(DISTINCT oc.customer_unique_id) AS customer_count,
SUM(CASE WHEN oor.review_score=5 THEN 1 END) AS five_star_reviews
FROM olist_orders oo
JOIN olist_customers oc ON oo.customer_id = oc.customer_id
JOIN olist_order_payments oop ON oo.order_id = oop.order_id
JOIN olist_order_items ooi ON oo.order_id = ooi.order_id
JOIN olist_order_reviews oor ON oo.order_id = oor.order_id
GROUP BY ooi.seller_id
ORDER BY goods_sold DESC, revenue DESC, customer_count DESC, five_star_reviews DESC
;
''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql.head()

,seller_id,goods_sold,revenue,customer_count,five_star_reviews
0,4a3ca9315b744ce9f8e9374361493884,2128,302403.81,1771,1021.0
1,6560211a19b47992c3666cc44a7e94c0,2111,178381.03,1809,1069.0
2,1f50f920176fa81dab994f9023523100,2009,290729.12,1383,1134.0
3,cc419e0650a3c5ba77189a1882b7556a,1885,143935.84,1649,1091.0
4,da8622b14eb17ae2831f4ac5b9dab84a,1656,275667.80,1270,948.0


In [10]:
# 9. Query 7: Delivery Success Rate Across States

#     Write and execute a SQL query to calculate the delivery success rate across different states.

# [ Strategy: Use similar GROUP BY with previous state calculation, without year.
# Where is delivery success rate found? oo.order_status [= delivered or something else]
# Delivery Success Rate =  SUM(CASE WHEN oo.order_status = 'delivered') * 1.00 / COUNT(oo.order_id) ]

sql='''
SELECT oc.customer_state,
SUM(CASE WHEN oo.order_status = 'delivered' THEN 1 END) * 1.00 / COUNT(oo.order_id) AS delivery_success_rate
FROM olist_orders oo
JOIN olist_customers oc ON oo.customer_id = oc.customer_id
GROUP BY oc.customer_state

;
''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,customer_state,delivery_success_rate
0,AC,0.987654
1,AL,0.961259
2,AM,0.979730
3,AP,0.985294
4,BA,0.963314
5,CE,0.957335
6,DF,0.971963
7,ES,0.981308
8,GO,0.968812
9,MA,0.959839


In [11]:
# 10. Query 8: Preferred Form of Payment for Different Categories

#     Write and execute a SQL query to find the preferred form of payment for different product categories.

#form of payment: oop.payment_type
# category: olist_products.product_category_name [JOIN olist_products op ON op.product_id = ooi.product_id]
#JOIN from oo through payment_type and category_name to link them both based in each line of olist_order_items ooi

sql='''
WITH PaymentCounts AS (
    SELECT 
        t.product_category_name_english AS category,
        p.payment_type,
        COUNT(*) AS use_count,
        RANK() OVER (
            PARTITION BY t.product_category_name_english 
            ORDER BY COUNT(*) DESC
        ) as rank_per_category
    FROM olist_order_items i
    JOIN olist_products_dataset pr ON i.product_id = pr.product_id
    JOIN product_category_name_translation t ON pr.product_category_name = t.product_category_name
    JOIN olist_order_payments p ON i.order_id = p.order_id
    GROUP BY category, p.payment_type
)
SELECT 
    category,
    payment_type AS top_payment_type,
    use_count
FROM PaymentCounts
WHERE rank_per_category = 1
ORDER BY use_count DESC;
''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

#Note that this can show several payment types if they are tied for top payment type.

,category,top_payment_type,use_count
0,bed_bath_table,credit_card,8959
1,health_beauty,credit_card,7566
2,sports_leisure,credit_card,6635
3,furniture_decor,credit_card,6379
4,computers_accessories,credit_card,5436
...,...,...,...
67,la_cuisine,credit_card,13
68,cds_dvds_musicals,credit_card,9
69,fashion_childrens_clothes,credit_card,5
70,security_and_services,credit_card,1


In [13]:
# 11. Query 9: Distance Between Cities

#     Write and execute a SQL query to calculate the distance between cities.

# This query will take a long time for all cities so we will add a LIMIT 10 to demonstrate the query while limiting the load time.


sql='''
WITH CityCoords AS (
    SELECT 
        geolocation_city,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng
    FROM olist_geolocation
    GROUP BY geolocation_city
)
SELECT 
    c1.geolocation_city AS city1,
    c2.geolocation_city AS city2,
    (acos(
        sin(c1.lat * PI() / 180) * sin(c2.lat * PI() / 180) +
        cos(c1.lat * PI() / 180) * cos(c2.lat * PI() / 180) *
        cos((c2.lng - c1.lng) * PI() / 180)
    ) * 6371) AS distance_km
FROM CityCoords c1, CityCoords c2
WHERE c1.geolocation_city < c2.geolocation_city
LIMIT 10
''';

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,city1,city2,distance_km
0,* cidade,...arraial do cabo,794.800452
1,* cidade,4o. centenario,403.405299
2,* cidade,4º centenario,403.576938
3,* cidade,abadia de goias,979.095260
4,* cidade,abadia dos dourados,813.737260
5,* cidade,abadiania,1044.797256
6,* cidade,abadiânia,1044.802850
7,* cidade,abaete,817.389142
8,* cidade,abaetetuba,2652.301760
9,* cidade,abaeté,817.592173
